# 5. Held-out SHAP data

Probability-scale SHAP and source data for Figure 5.


## Load fitted models

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "mea_ml" / "io.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
from IPython.display import display
from typing import List, Tuple
from mea_ml.io import ROOT as PROJECT_ROOT, load_analysis

OUT_DIR = PROJECT_ROOT / "results/05_interpretation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
bundle, artifact, lineage = load_analysis()
RANDOM_STATE = artifact["random_state"]

import shap
import catboost

SHAP_PERMUTATIONS = 10


## Selected model and held-out predictions

In [3]:
X_map = {
    "mean+delta": bundle["X_mean_delta_df"].copy(),
    "delta-only": bundle["X_delta_only_df"].copy(),
    "mfr-only": bundle["X_mfr_only_df"].copy(),
}
feature_name_map = {
    "mean+delta": list(bundle["feature_names_mean_delta"]),
    "delta-only": list(bundle["feature_names_delta_only"]),
    "mfr-only": list(bundle["feature_names_mfr_only"]),
}
y = np.asarray(bundle["y"]).astype(int)
groups = np.asarray(bundle["groups"]).astype(str)
paired_meta = bundle["paired_meta"].copy()
feature_manifest = bundle["feature_manifest"].copy()

selected_model = str(artifact["selected_model_name"])
selected_feature_set = str(artifact["selected_feature_set"])

X_primary = X_map[selected_feature_set].copy()
feature_names_primary = feature_name_map[selected_feature_set]
assert len(X_primary) == len(y) == len(groups) == len(paired_meta)

outer_fitted_models = artifact["outer_fitted_models"]
key = (selected_model, selected_feature_set)
if key not in outer_fitted_models:
    raise KeyError(f"Selected primary combination not found in outer_fitted_models: {key}")

selected_fold_models = outer_fitted_models[key]
N_REPEATS = int(artifact["n_repeats"])
OUTER_N_SPLITS = int(artifact["outer_n_splits"])
assert len(selected_fold_models) == N_REPEATS * OUTER_N_SPLITS
saved_repeat_oof = artifact["oof_predictions_by_repeat"].copy()
saved_repeat_oof = saved_repeat_oof[
    (saved_repeat_oof["model_name"] == selected_model) &
    (saved_repeat_oof["feature_set"] == selected_feature_set)
].copy()
assert not saved_repeat_oof.duplicated(["repeat_id", "sample_index"]).any()
print("N outer fold models:", len(selected_fold_models))
print("Primary X shape:", X_primary.shape)


N outer fold models: 15
Primary X shape: (90, 88)


## Feature families

In [4]:
raw_feature_to_family = dict(zip(feature_manifest["raw_feature"], feature_manifest["feature_family"]))

def engineered_feature_to_family(name):
    return raw_feature_to_family[name.split("::", 1)[1]]

primary_feature_family_map = pd.DataFrame({
    "feature_name": feature_names_primary,
    "feature_family": [engineered_feature_to_family(f) for f in feature_names_primary],
})
primary_feature_family_map.to_csv(OUT_DIR / "feature_family_map_primary.csv", index=False)


## Held-out probability SHAP

In [5]:
def subset_rows_by_groups(X_df: pd.DataFrame, groups_arr: np.ndarray, group_subset: List[str]) -> Tuple[pd.DataFrame, np.ndarray]:
    mask = np.isin(groups_arr, np.array(group_subset, dtype=str))
    return X_df.loc[mask].reset_index(drop=True), mask

def transformed_feature_names_from_pipeline(pipe, fallback_columns: List[str]) -> List[str]:
    if "corr" in pipe.named_steps:
        return list(pipe.named_steps["corr"].selected_columns_)
    return list(fallback_columns)

all_shap_rows = []
prediction_audit_rows = []
explainer_audit_rows = []

for fm in selected_fold_models:
    outer_fold_id = int(fm["outer_fold_id"])
    repeat_id = int(fm.get("repeat_id", (outer_fold_id - 1) // OUTER_N_SPLITS + 1))
    fold_in_repeat = int(fm.get("fold_in_repeat", (outer_fold_id - 1) % OUTER_N_SPLITS + 1))
    test_groups = list(fm["test_groups"])
    train_groups = list(fm["train_groups"])
    assert set(test_groups).isdisjoint(train_groups)
    pipe = fm["estimator"]

    X_test, test_mask = subset_rows_by_groups(X_primary, groups, test_groups)
    X_train, train_mask = subset_rows_by_groups(X_primary, groups, train_groups)
    assert not np.any(test_mask & train_mask)
    sample_indices = np.flatnonzero(test_mask)
    y_test = y[test_mask]
    meta_test = paired_meta.loc[test_mask].reset_index(drop=True).copy()

    X_trans = X_test.copy()
    X_background = X_train.copy()
    for name in ["corr", "scaler"]:
        if name in pipe.named_steps:
            X_trans = pipe.named_steps[name].transform(X_trans)
            X_background = pipe.named_steps[name].transform(X_background)
    transformed_names = transformed_feature_names_from_pipeline(pipe, list(X_test.columns))
    X_trans = pd.DataFrame(X_trans, columns=transformed_names)
    X_background = pd.DataFrame(X_background, columns=transformed_names)
    clf = pipe.named_steps["clf"]
    positive_index = list(clf.classes_).index(1)

    def positive_probability(z):
        z_df = pd.DataFrame(z, columns=transformed_names)
        return clf.predict_proba(z_df)[:, positive_index]

    masker = shap.maskers.Independent(X_background, max_samples=len(X_background))
    explainer = shap.Explainer(
        positive_probability, masker, algorithm="permutation",
        seed=RANDOM_STATE + outer_fold_id,
    )
    sv = explainer(
        X_trans,
        max_evals=SHAP_PERMUTATIONS * (2 * X_trans.shape[1] + 1),
        silent=True,
    )
    shap_values = np.asarray(sv.values, dtype=float)
    base_values = np.asarray(sv.base_values, dtype=float).reshape(-1)
    assert shap_values.shape == X_trans.shape
    y_proba = pipe.predict_proba(X_test)[:, positive_index]
    reconstructed = base_values + shap_values.sum(axis=1)
    assert np.allclose(reconstructed, y_proba, atol=1e-5, rtol=1e-5)

    saved = saved_repeat_oof[
        (saved_repeat_oof["repeat_id"] == repeat_id) &
        (saved_repeat_oof["outer_fold_id"] == outer_fold_id)
    ].set_index("sample_index").loc[sample_indices]
    assert np.array_equal(saved["y_true"].to_numpy(), y_test)
    assert np.allclose(saved["y_proba"], y_proba, atol=1e-6, rtol=1e-6)
    full_shap = pd.DataFrame(shap_values, columns=transformed_names).reindex(columns=feature_names_primary, fill_value=0.0)
    explainer_audit_rows.append({
        "outer_fold_id": outer_fold_id, "repeat_id": repeat_id,
        "method": "PermutationExplainer", "output_space": "positive_probability",
        "n_background": len(X_background), "n_test": len(X_trans),
        "background_groups": "|".join(train_groups), "test_groups": "|".join(test_groups),
        "permutation_cycles_budget": SHAP_PERMUTATIONS,
        "max_evals_per_sample": SHAP_PERMUTATIONS * (2 * X_trans.shape[1] + 1),
        "n_retained_features": len(transformed_names),
        "max_additivity_error": float(np.max(np.abs(reconstructed - y_proba))),
        "max_saved_prediction_error": float(np.max(np.abs(saved["y_proba"].to_numpy() - y_proba))),
    })
    for i, sample_index in enumerate(sample_indices):
        prediction_audit_rows.append({
            "sample_index": int(sample_index), "repeat_id": repeat_id,
            "outer_fold_id": outer_fold_id, "group_id": str(groups[sample_index]),
            "well": str(meta_test.loc[i, "well"]), "y_true": int(y_test[i]),
            "base_value": float(base_values[i]), "shap_sum": float(full_shap.iloc[i].sum()),
            "y_proba": float(y_proba[i]), "reconstructed_probability": float(reconstructed[i]),
        })
        for j, feat in enumerate(feature_names_primary):
            all_shap_rows.append({
                "outer_fold_id": outer_fold_id,
                "repeat_id": repeat_id,
                "fold_in_repeat": fold_in_repeat,
                "sample_index": int(sample_index),
                "sample_row_within_fold": i,
                "group_id": str(meta_test.loc[i, "group_id"]),
                "class": str(meta_test.loc[i, "class"]),
                "pair_stem": str(meta_test.loc[i, "pair_stem"]),
                "well": str(meta_test.loc[i, "well"]),
                "y_true": int(y_test[i]),
                "feature_name": feat,
                "feature_value": float(X_test.loc[i, feat]),
                "shap_value": float(full_shap.iloc[i, j]),
                "selected_in_fold": feat in transformed_names,
                "feature_family": engineered_feature_to_family(feat),
            })
    print(f"SHAP repeat {repeat_id}/{N_REPEATS}, fold {fold_in_repeat}/{OUTER_N_SPLITS}: {len(X_trans)} held-out wells", flush=True)

shap_long_df = pd.DataFrame(all_shap_rows)
prediction_audit_df = pd.DataFrame(prediction_audit_rows)
explainer_audit_df = pd.DataFrame(explainer_audit_rows)
assert not prediction_audit_df.duplicated(["repeat_id", "sample_index"]).any()
assert prediction_audit_df.groupby("sample_index").size().eq(N_REPEATS).all()
assert prediction_audit_df.groupby("repeat_id").size().eq(len(y)).all()
assert len(shap_long_df) == len(y) * N_REPEATS * len(feature_names_primary)

shap_mean_df = (
    shap_long_df.groupby(["sample_index", "group_id", "class", "pair_stem", "well", "y_true", "feature_name", "feature_family"], as_index=False)
    .agg(feature_value=("feature_value", "first"), shap_value=("shap_value", "mean"), n_repeats=("repeat_id", "nunique"))
)
mean_prediction_audit = prediction_audit_df.groupby("sample_index", as_index=False).agg(
    base_value=("base_value", "mean"), y_proba=("y_proba", "mean"),
    reconstructed_probability=("reconstructed_probability", "mean"),
)
mean_shap_sums = shap_mean_df.groupby("sample_index")["shap_value"].sum().sort_index()
assert np.allclose(mean_prediction_audit["base_value"] + mean_shap_sums.to_numpy(), mean_prediction_audit["y_proba"], atol=1e-5)
saved_mean = artifact["oof_predictions_main"]
saved_mean = saved_mean[(saved_mean["model_name"] == selected_model) & (saved_mean["feature_set"] == selected_feature_set)].sort_values("sample_index")
assert np.array_equal(mean_prediction_audit["sample_index"], saved_mean["sample_index"])
assert np.allclose(mean_prediction_audit["y_proba"], saved_mean["y_proba"], atol=1e-6)

print("SHAP long rows:", shap_long_df.shape)
print("Unique well pairs:", shap_mean_df["sample_index"].nunique())


SHAP repeat 1/3, fold 1/5: 18 held-out wells
SHAP repeat 1/3, fold 2/5: 18 held-out wells
SHAP repeat 1/3, fold 3/5: 18 held-out wells
SHAP repeat 1/3, fold 4/5: 18 held-out wells
SHAP repeat 1/3, fold 5/5: 18 held-out wells
SHAP repeat 2/3, fold 1/5: 18 held-out wells
SHAP repeat 2/3, fold 2/5: 18 held-out wells
SHAP repeat 2/3, fold 3/5: 18 held-out wells
SHAP repeat 2/3, fold 4/5: 18 held-out wells
SHAP repeat 2/3, fold 5/5: 18 held-out wells
SHAP repeat 3/3, fold 1/5: 18 held-out wells
SHAP repeat 3/3, fold 2/5: 18 held-out wells
SHAP repeat 3/3, fold 3/5: 18 held-out wells
SHAP repeat 3/3, fold 4/5: 18 held-out wells
SHAP repeat 3/3, fold 5/5: 18 held-out wells
SHAP long rows: (23760, 15)
Unique well pairs: 90


## Rank features for Figure 5


In [6]:
shap_feature_ranking = (
    shap_mean_df.groupby(["feature_name", "feature_family"], as_index=False)
    .agg(mean_abs_shap=("shap_value", lambda s: float(np.mean(np.abs(s)))))
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
display(shap_feature_ranking.head(12))


,feature_name,feature_family,mean_abs_shap
0,Mean::Area Under Normalized Cross-Correlation,synchrony_correlation,0.147622
1,Delta::Width at Half Height of Normalized Cros...,synchrony_correlation,0.062451
2,Mean::Median ISI within Burst - Avg,burst_timing,0.039510
3,Mean::Number of Active Electrodes,spike_activity,0.027846
4,Delta::Network ISI Coefficient of Variation,burst_timing,0.016598
5,Mean::Network ISI Coefficient of Variation,burst_timing,0.011975
6,Mean::Weighted Mean Firing Rate (Hz),spike_activity,0.009480
7,Mean::Normalized Duration IQR - Avg,burst_timing,0.007910
8,Delta::Width at Half Height of Cross-Correlation,synchrony_correlation,0.006969
9,Mean::Burst Percentage - Std,burst,0.006700


## Save explanations

In [7]:
shap_feature_ranking.to_csv(OUT_DIR / "shap_feature_ranking.csv", index=False)
shap_long_df.to_csv(OUT_DIR / "shap_values_oof.csv", index=False)
shap_mean_df.to_csv(OUT_DIR / "shap_values_oof_mean.csv", index=False)
prediction_audit_df.to_csv(OUT_DIR / "shap_prediction_audit_by_repeat.csv", index=False)
mean_prediction_audit.to_csv(OUT_DIR / "shap_prediction_audit_mean.csv", index=False)
explainer_audit_df.to_csv(OUT_DIR / "shap_explainer_audit.csv", index=False)

summary = {
    "lineage": lineage,
    "training_random_state": RANDOM_STATE,
    "selected_model_name": selected_model,
    "selected_feature_set": selected_feature_set,
    "n_outer_folds": int(len(selected_fold_models)),
    "n_repeats": N_REPEATS,
    "n_outer_folds_per_repeat": OUTER_N_SPLITS,
    "n_unique_well_pairs": len(y),
    "n_heldout_evaluations": len(prediction_audit_df),
    "shap_method": "PermutationExplainer, interventional masking, positive-class probability",
    "shap_permutation_cycles_budget": SHAP_PERMUTATIONS,
    "shap_max_evals_per_sample": "10 * (2 * number of retained features + 1)",
    "shap_background": "all corresponding outer-training wells; fitted preprocessing only",
    "shap_summary_unit": "one explanation per well, averaged across repeats",
    "excluded_features": "zero contribution in folds where the feature was filtered out",
    "max_additivity_error": float(explainer_audit_df["max_additivity_error"].max()),
    "max_saved_prediction_error": float(explainer_audit_df["max_saved_prediction_error"].max()),
    "library_versions": {"shap": shap.__version__, "numpy": np.__version__, "pandas": pd.__version__, "catboost": catboost.__version__},
    "n_shap_rows": int(len(shap_long_df)),
    "n_top_feature_rows": int(len(shap_feature_ranking)),
}
with open(OUT_DIR / "interpretation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved SHAP interpretation outputs to:", OUT_DIR.relative_to(PROJECT_ROOT))


Saved SHAP interpretation outputs to: results/05_interpretation
